This code reads the whole dataframe and prediction files.

It generates table "mismatches_merged.csv" that contains columns:
"Lab_Number,Prediction,Overall_Assessment_x,Overall_Interpretation,Overall_Assessment_y"

Columns model "Prediction" and "Overall_Assesment_x" (ground_truth) are integer form.
Column "Overall_Assesment" is a long sample description string.

The goal is to analyze this string and detect cases in which there is misalignment in the ground truth class detection and the sample description.

Right now the analysis is made only on the samples where model was wrong (hence "mismatches"). Similar analysis could be made for all samples.

In [3]:
import pandas as pd

In [8]:
import pandas as pd
from pathlib import Path

# -----------------------------
# 1. Load prediction files
# -----------------------------
predictions_dict = {}

for file_path in sorted(Path(".").glob("predictions_*.csv")):
    suffix = file_path.stem.replace("predictions_", "")
    key = int(suffix) if suffix.isdigit() else suffix

    df = pd.read_csv(
        file_path,
        usecols=["Lab_Number", "Prediction", "Overall_Assessment"],
        sep=None,
        engine="python"
    )

    df["Overall_Assessment"] = pd.to_numeric(
        df["Overall_Assessment"],
        errors="coerce"
    ).astype("Int64")

    predictions_dict[key] = df

# -----------------------------
# 2. Prepare outputs
# -----------------------------
all_rows_dict = {}
mismatches_dict = {}

final_columns = [
    "Probe",
    "Lab_Number",
    "Prediction",
    "Overall_Assessment_pred_file",
    "Overall_Interpretation",
    "Overall_Assessment_data",
]

# -----------------------------
# 3. Build per-probe dataframes
# -----------------------------
for key, df in predictions_dict.items():
    merged = df.merge(
        data[["Lab_Number", "Overall_Interpretation", "Overall_Assessment"]],
        on="Lab_Number",
        how="left",
        suffixes=("_pred_file", "_data")
    )

    merged["Probe"] = key

    # same column order everywhere
    merged = merged.reindex(columns=final_columns)

    # save all rows
    all_rows_dict[key] = merged.copy()

    # save mismatches only
    mismatch = merged[
        merged["Prediction"] != merged["Overall_Assessment_pred_file"]
    ].copy().reset_index(drop=True)

    mismatches_dict[key] = mismatch

# -----------------------------
# 4. Save per-probe files
# -----------------------------
output_dir = Path("../probe_outputs")
output_dir.mkdir(exist_ok=True)

for key, df_all in all_rows_dict.items():
    df_all.to_csv(output_dir / f"all_rows_{key}.csv", index=False)

for key, df_mismatch in mismatches_dict.items():
    df_mismatch.to_csv(output_dir / f"mismatches_{key}.csv", index=False)

# -----------------------------
# 5. Save merged files
# -----------------------------
all_rows_merged = pd.concat(all_rows_dict.values(), ignore_index=True)
mismatches_merged = pd.concat(mismatches_dict.values(), ignore_index=True)

all_rows_merged = all_rows_merged.reindex(columns=final_columns)
mismatches_merged = mismatches_merged.reindex(columns=final_columns)

all_rows_merged.to_csv(output_dir / "all_rows_merged.csv", index=False)
mismatches_merged.to_csv(output_dir / "mismatches_merged.csv", index=False)

# optional preview
print("Saved files to:", output_dir.resolve())
print("All rows merged shape:", all_rows_merged.shape)
print("Mismatches merged shape:", mismatches_merged.shape)

Saved files to: /Users/jeremiasz/praca/ecol-czyszczenie/probe_outputs
All rows merged shape: (18848, 6)
Mismatches merged shape: (3630, 6)


In [6]:
predictions_dict

{1:      Lab_Number  Prediction  Overall_Assessment
 0      P1300010           0                   1
 1      P1304884           0                   0
 2      P1400003           2                   2
 3      P1400044           2                   2
 4      P1400113           0                   0
 ...         ...         ...                 ...
 3701   P1802727           0                   0
 3702   P1802064           0                   1
 3703   P1802226           0                   0
 3704   P1802068           2                   2
 3705   P1802065           1                   1
 
 [3706 rows x 3 columns],
 2:      Lab_Number  Prediction  Overall_Assessment
 0      P1802203           2                   1
 1      P1802122           2                   2
 2      P1802153           0                   0
 3      P1802139           1                   1
 4      P1802217           2                   2
 ...         ...         ...                 ...
 3675   P2003760           0       

In [7]:
from pathlib import Path

folder = Path("../mismatches")
out_path = folder / "mismatches_merged.csv"

if out_path.exists():
    out_path.unlink()


files = sorted(folder.glob("*.csv"))

with out_path.open("w", encoding="utf-8", newline="") as out:
    for i, f in enumerate(files):
        with f.open("r", encoding="utf-8") as inp:
            if i > 0:
                next(inp, None)
            out.writelines(inp)